# Relay: Qwen3-Coder vLLM Backend Runbook (Kaggle Dual T4)

This notebook provides the **canonical, reproducible runbook** to deploy `QuantTrio/Qwen3-Coder-30B-A3B-Instruct-AWQ` on a Kaggle Notebook with **2× NVIDIA Tesla T4 GPUs** using **vLLM 0.29.0**, and expose it to the **Relay** LLM gateway over a Cloudflare Quick Tunnel.

### Prerequisites (Kaggle Notebook Settings)
Before running the cells below, verify your notebook settings in the right sidebar:
1. **Accelerator**: `GPU T4 x2` (Requires phone verification on Kaggle)
2. **Internet**: `On` (Required to download weights and run the Cloudflare Tunnel)
3. **Persistence**: `No persistence` or `Variables only`
4. **Environment**: `Always use latest environment`

---
### Execution Flow
1. Verify dual T4 GPUs and CUDA 13 environment.
2. Verify vLLM 0.29.0 and model checkpoint access.
3. Clean up stale worker processes if restarting.
4. Launch vLLM in the background and poll `/v1/models` for readiness.
5. Validate local inference (`PONG` deterministic check and streaming).
6. Start Cloudflare Quick Tunnel and verify public ingress.
7. Output Relay `.env` configuration snippet.

## 1. Environment & Path Setup
**Purpose**: Verify Python version, OS platform, and ensure NVIDIA CUDA 13 shared libraries are prepended to `LD_LIBRARY_PATH`.  
**Expected Result**: Python 3.12+, Linux x86_64, CUDA 13 path verified.  
**Failure Action**: If Python is missing or CUDA path is invalid, verify Kaggle environment settings.

In [ ]:
import os
import sys
import platform

print(f"Python: {sys.version.split()[0]} ({platform.python_implementation()})")
print(f"OS: {platform.system()} {platform.release()} ({platform.machine()})")

# Ensure CUDA 13 shared library path is set for Kaggle Python 3.12 environment
cu13_path = "/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib"
if os.path.exists(cu13_path):
    current_ld = os.environ.get("LD_LIBRARY_PATH", "")
    if cu13_path not in current_ld:
        os.environ["LD_LIBRARY_PATH"] = f"{cu13_path}:{current_ld}"
        print(f"Configured LD_LIBRARY_PATH with CUDA 13 libraries: {cu13_path}")
else:
    print(f"Note: Standard cu13 path not detected; using default library paths.")

print(f"Working Directory: {os.getcwd()}")
print("Environment Setup: PASS")

## 2. GPU Hardware Verification
**Purpose**: Run `nvidia-smi` and verify that exactly two NVIDIA Tesla T4 GPUs are attached with ~15GB VRAM each.  
**Expected Result**: 2 GPUs reported: `Tesla T4`, ~15,109 MiB free each.  
**Failure Action**: If fewer than 2 GPUs are found, open the right sidebar and set Accelerator to `GPU T4 x2`.

In [ ]:
import subprocess

try:
    smi_out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=index,name,memory.total,memory.free,driver_version", "--format=csv,noheader"],
        text=True
    ).strip().splitlines()
    
    print(f"Detected {len(smi_out)} GPU(s):")
    for line in smi_out:
        print(f"  GPU {line}")
        
    assert len(smi_out) >= 2, f"Expected 2 GPUs for tensor-parallel-size=2, but found {len(smi_out)}."
    print("\nGPU Hardware Verification: PASS (2× Tesla T4 detected)")
except Exception as e:
    print(f"GPU Hardware Verification: FAIL - {e}")
    print("Action Required: In Kaggle Settings (right sidebar), change Accelerator to 'GPU T4 x2'.")
    raise

## 3. CUDA & PyTorch Allocation Verification
**Purpose**: Test CUDA runtime availability and perform a test memory allocation on both `cuda:0` and `cuda:1`.  
**Expected Result**: `torch.cuda.is_available() == True`, device count $\ge 2$, allocation succeeds on both devices.  
**Failure Action**: If CUDA fails, restart the notebook session.

In [ ]:
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
assert torch.cuda.is_available(), "CUDA is not available in PyTorch!"

device_count = torch.cuda.device_count()
print(f"CUDA Device Count: {device_count}")
assert device_count >= 2, f"Need at least 2 CUDA devices, found {device_count}."

for i in range(device_count):
    name = torch.cuda.get_device_name(i)
    free_mem, total_mem = torch.cuda.mem_get_info(i)
    free_mb = free_mem / (1024**2)
    total_mb = total_mem / (1024**2)
    print(f"  Device {i} ({name}): {free_mb:.0f} MiB free / {total_mb:.0f} MiB total")
    
    # Tensor allocation sanity check
    test_tensor = torch.zeros((1024, 1024), device=f"cuda:{i}", dtype=torch.float16)
    del test_tensor

torch.cuda.empty_cache()
print("\nCUDA & PyTorch Hardware Sanity: PASS")

## 4. vLLM Installation & CLI Verification
**Purpose**: Verify vLLM is installed and the canonical `vllm serve` CLI command is available.  
**Expected Result**: vLLM version (e.g. 0.29.0) and `vllm serve --help` exit code 0.  
**Note**: The canonical command is `vllm serve` (NOT `vllm server`).

In [ ]:
import subprocess
import shutil

vllm_bin = shutil.which("vllm")
print(f"vLLM Executable: {vllm_bin}")

if not vllm_bin:
    print("vLLM binary not found on PATH. Installing vLLM...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "vllm"])
    vllm_bin = shutil.which("vllm")

# Check version
try:
    import vllm
    print(f"vLLM Python Package Version: {vllm.__version__}")
except ImportError:
    print("Notice: vllm module not imported in current kernel process; CLI check will proceed.")

# Verify `vllm serve` CLI availability
help_check = subprocess.run(["vllm", "serve", "--help"], capture_output=True, text=True)
if help_check.returncode == 0:
    print("CLI Command 'vllm serve': AVAILABLE (PASS)")
else:
    print(f"CLI check warning: {help_check.stderr[:300]}")
    raise RuntimeError("vllm serve CLI command is not available.")

## 5. Model Repository & Metadata Verification
**Purpose**: Verify that the exact target model checkpoint `QuantTrio/Qwen3-Coder-30B-A3B-Instruct-AWQ` is accessible on Hugging Face.  
**Expected Result**: Repository exists, architecture is Qwen3-MoE, quantization is AWQ 4-bit.

In [ ]:
from huggingface_hub import HfApi

MODEL_ID = "QuantTrio/Qwen3-Coder-30B-A3B-Instruct-AWQ"
api = HfApi()

try:
    info = api.model_info(MODEL_ID)
    print(f"Hugging Face Model: {info.id}")
    print(f"Model Architecture: {info.config.get('architectures', ['Unknown'])}")
    print(f"Quantization:       {info.config.get('quantization_config', {})}")
    print("Model Accessibility Check: PASS")
except Exception as e:
    print(f"Model accessibility check failed: {e}")
    print("Ensure Kaggle notebook has Internet enabled in right sidebar settings.")
    raise

## 6. Safe GPU & Process Cleanup
**Purpose**: Check if port 8000 is held or stale vLLM processes are running from a previous run. Only terminate matching stale processes to recover GPU VRAM.  
**Expected Result**: Port 8000 free and both GPUs clean (~15GB free each).

In [ ]:
import os
import signal
import subprocess
import time

PORT = 8000
cleaned = False

# 1. Check port 8000 occupancy
try:
    lsof_out = subprocess.check_output(["lsof", "-ti", f":{PORT}"], text=True).strip().split()
    for pid_str in lsof_out:
        pid = int(pid_str)
        if pid != os.getpid():
            print(f"Terminating stale process {pid} holding port {PORT}...")
            os.kill(pid, signal.SIGTERM)
            cleaned = True
except (subprocess.CalledProcessError, FileNotFoundError):
    print(f"Port {PORT} is clean (no active listener).")

# 2. Check stale vLLM processes
try:
    pgrep_out = subprocess.check_output(["pgrep", "-f", "vllm"], text=True).strip().split()
    for pid_str in pgrep_out:
        pid = int(pid_str)
        if pid != os.getpid():
            print(f"Terminating stale vLLM process (PID {pid})...")
            try:
                os.kill(pid, signal.SIGTERM)
                cleaned = True
            except ProcessLookupError:
                pass
except (subprocess.CalledProcessError, FileNotFoundError):
    print("No stale vLLM processes detected.")

if cleaned:
    print("Waiting 5 seconds for GPU memory release...")
    time.sleep(5)

# Verify VRAM status
smi_mem = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=index,memory.used,memory.free", "--format=csv,noheader"],
    text=True
).strip()
print("\nCurrent GPU Memory:")
print(smi_mem)

## 7. Start vLLM in Background
**Purpose**: Launch vLLM using the **canonical verified configuration**. Output logs are written to `/kaggle/working/vllm_server.log`.  
**Critical Flags**:
- `--model QuantTrio/Qwen3-Coder-30B-A3B-Instruct-AWQ`
- `--served-model-name qwen3-coder-30b` (Single value, no comma-separated aliases)
- `--tensor-parallel-size 2`
- `--dtype float16` (Required for Turing T4)
- `--quantization awq`
- `--max-model-len 4096`
- `--max-num-seqs 4`
- `--gpu-memory-utilization 0.85`
- `--enforce-eager`
- `--trust-remote-code`
- Note: Do NOT add `--swap-space` (not supported in vLLM 0.29.0)

In [ ]:
import subprocess
import os
import time

MODEL_ID = "QuantTrio/Qwen3-Coder-30B-A3B-Instruct-AWQ"
SERVED_MODEL_NAME = "qwen3-coder-30b"
LOG_FILE = "/kaggle/working/vllm_server.log"
PID_FILE = "/kaggle/working/vllm.pid"

# Canonical command
vllm_cmd = [
    "vllm", "serve", MODEL_ID,
    "--served-model-name", SERVED_MODEL_NAME,
    "--host", "0.0.0.0",
    "--port", "8000",
    "--tensor-parallel-size", "2",
    "--dtype", "float16",
    "--quantization", "awq",
    "--max-model-len", "4096",
    "--max-num-seqs", "4",
    "--gpu-memory-utilization", "0.85",
    "--enforce-eager",
    "--trust-remote-code"
]

print("Launching vLLM in background:")
print(" ".join(vllm_cmd))

# Clear previous log
if os.path.exists(LOG_FILE):
    os.remove(LOG_FILE)

log_fp = open(LOG_FILE, "w")

# Prepare environment with CUDA 13 path
env = os.environ.copy()
cu13 = "/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib"
if os.path.exists(cu13):
    env["LD_LIBRARY_PATH"] = f"{cu13}:{env.get('LD_LIBRARY_PATH', '')}"

proc = subprocess.Popen(
    vllm_cmd,
    stdout=log_fp,
    stderr=subprocess.STDOUT,
    env=env,
    preexec_fn=os.setpgrp
)

with open(PID_FILE, "w") as f:
    f.write(str(proc.pid))

print(f"\nvLLM process started (PID: {proc.pid})")
print(f"Logs streaming to: {LOG_FILE}")
time.sleep(4)

# Quick sanity check that process hasn't immediately died
if proc.poll() is not None:
    print(f"ERROR: vLLM exited immediately with code {proc.returncode}!")
    with open(LOG_FILE) as f:
        print(f.read()[-3000:])
    raise RuntimeError("vLLM startup failure. See log above.")
else:
    print("Initial startup check: PASS (Process is running)")

## 8. Wait for vLLM Readiness Probe
**Purpose**: Poll `http://127.0.0.1:8000/v1/models` every 3 seconds until the server responds with HTTP 200.  
**Expected Duration**: ~2–3 minutes (downloading cached weights and initializing the KV cache).  
**Failure Action**: If it times out after 5 minutes, inspect `vllm_server.log` printed below.

In [ ]:
import urllib.request
import json
import time

HEALTH_URL = "http://127.0.0.1:8000/v1/models"
MAX_WAIT_SECONDS = 360 # 6 minutes max
start_time = time.time()
ready = False

print("Waiting for vLLM to load weights and initialize KV cache...")

while time.time() - start_time < MAX_WAIT_SECONDS:
    try:
        req = urllib.request.Request(HEALTH_URL)
        with urllib.request.urlopen(req, timeout=3) as resp:
            if resp.status == 200:
                data = json.loads(resp.read().decode())
                elapsed = int(time.time() - start_time)
                print(f"\nSUCCESS: vLLM is ONLINE and healthy after {elapsed}s!")
                print(f"Registered Models: {[m['id'] for m in data.get('data', [])]}")
                ready = True
                break
    except Exception:
        elapsed = int(time.time() - start_time)
        if elapsed % 15 == 0 and elapsed > 0:
            print(f"  Still loading... ({elapsed}s elapsed)")
        time.sleep(3)

if not ready:
    print("\nTIMEOUT: vLLM did not become healthy within the deadline.")
    print("Last 40 lines of vllm_server.log:")
    with open("/kaggle/working/vllm_server.log") as f:
        print("".join(f.readlines()[-40:]))
    raise TimeoutError("vLLM readiness timeout.")

## 9. Local `/v1/models` Verification
**Purpose**: Verify that the served model ID is clean (`qwen3-coder-30b`).  
**Expected Result**: HTTP 200 response containing `qwen3-coder-30b`.

In [ ]:
import urllib.request
import json

req = urllib.request.Request("http://127.0.0.1:8000/v1/models")
with urllib.request.urlopen(req, timeout=5) as resp:
    models_data = json.loads(resp.read().decode())

print("GET http://127.0.0.1:8000/v1/models (HTTP 200):")
print(json.dumps(models_data, indent=2))

registered_ids = [m["id"] for m in models_data.get("data", [])]
assert "qwen3-coder-30b" in registered_ids, f"Expected 'qwen3-coder-30b', found {registered_ids}"
print("\nLocal Model Listing Check: PASS")

## 10. Local Inference Test (Deterministic PONG)
**Purpose**: Send a non-streaming test completion request with `temperature: 0.0` requesting the single word `PONG`.  
**Expected Result**: Response content contains `PONG`.

In [ ]:
import urllib.request
import json

payload = {
    "model": "qwen3-coder-30b",
    "messages": [
        {"role": "user", "content": "Reply with only the single uppercase word PONG"}
    ],
    "temperature": 0.0,
    "max_tokens": 16
}

req = urllib.request.Request(
    "http://127.0.0.1:8000/v1/chat/completions",
    data=json.dumps(payload).encode("utf-8"),
    headers={"Content-Type": "application/json"}
)

with urllib.request.urlopen(req, timeout=60) as resp:
    res = json.loads(resp.read().decode())

content = res["choices"][0]["message"]["content"].strip()
print("Response:")
print(f"  Content:    '{content}'")
print(f"  Token Usage: {res.get('usage')}")

assert "PONG" in content.upper(), f"Expected 'PONG' in response, got: '{content}'"
print("\nLocal Non-Streaming Inference Test: PASS")

## 11. Local Streaming Test (Server-Sent Events)
**Purpose**: Test `stream: true` to verify that chunked token delivery works without buffering.  
**Expected Result**: Streamed token deltas printed incrementally to stdout.

In [ ]:
import urllib.request
import json

stream_payload = {
    "model": "qwen3-coder-30b",
    "stream": True,
    "messages": [
        {"role": "user", "content": "Write a 1-line Python lambda function to reverse a string."}
    ],
    "temperature": 0.2,
    "max_tokens": 64
}

req = urllib.request.Request(
    "http://127.0.0.1:8000/v1/chat/completions",
    data=json.dumps(stream_payload).encode("utf-8"),
    headers={"Content-Type": "application/json"}
)

print("Streaming Tokens from vLLM:")
tokens = []

with urllib.request.urlopen(req, timeout=60) as resp:
    for line in resp:
        line_str = line.decode("utf-8").strip()
        if line_str.startswith("data: ") and line_str != "data: [DONE]":
            chunk_json = json.loads(line_str[6:])
            delta = chunk_json["choices"][0]["delta"].get("content", "")
            tokens.append(delta)
            print(delta, end="", flush=True)

full_code = "".join(tokens).strip()
print(f"\n\nComplete Output: {full_code}")
assert len(full_code) > 0, "No tokens received from stream!"
print("\nLocal Streaming Test: PASS")

## 12. cloudflared Binary Verification
**Purpose**: Check whether `cloudflared` is installed at `/kaggle/working/cloudflared`. If missing, automatically download the official Linux amd64 binary.  
**Expected Result**: `cloudflared` executable found and version reported.

In [ ]:
import os
import subprocess

CF_BIN = "/kaggle/working/cloudflared"

if not os.path.exists(CF_BIN) or not os.access(CF_BIN, os.X_OK):
    print(f"Downloading official cloudflared binary to {CF_BIN}...")
    subprocess.check_call([
        "wget", "-q", "-nc",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", CF_BIN
    ])
    os.chmod(CF_BIN, 0o755)

ver_out = subprocess.check_output([CF_BIN, "--version"], text=True).strip()
print(f"Binary Path:    {CF_BIN}")
print(f"Binary Version: {ver_out}")
print("cloudflared Verification: PASS")

## 13. Start Cloudflare Quick Tunnel in Background
**Purpose**: Start `cloudflared tunnel --url http://127.0.0.1:8000` in the background, redirecting logs to `/kaggle/working/cloudflared.log`, and dynamically extract the unique `trycloudflare.com` URL.  
**Expected Result**: A valid `https://<random-id>.trycloudflare.com` URL is extracted.

In [ ]:
import os
import subprocess
import time
import re

CF_BIN = "/kaggle/working/cloudflared"
CF_LOG = "/kaggle/working/cloudflared.log"
CF_PID = "/kaggle/working/cloudflared.pid"

if os.path.exists(CF_LOG):
    os.remove(CF_LOG)

proc = subprocess.Popen(
    [CF_BIN, "tunnel", "--url", "http://127.0.0.1:8000", "--logfile", CF_LOG],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    preexec_fn=os.setpgrp
)

with open(CF_PID, "w") as f:
    f.write(str(proc.pid))

print(f"cloudflared tunnel process started (PID: {proc.pid})")
print("Waiting up to 30s for dynamic public URL...")

tunnel_url = None
for _ in range(30):
    time.sleep(1)
    if os.path.exists(CF_LOG):
        with open(CF_LOG) as f:
            content = f.read()
            matches = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
            if matches:
                tunnel_url = matches[-1]
                break

assert tunnel_url is not None, "Failed to extract Cloudflare Tunnel URL within 30 seconds."
print("\n========================================================")
print(f"PUBLIC CLOUDFLARE TUNNEL URL: {tunnel_url}")
print("========================================================")

## 14. Public Ingress Test (`GET /v1/models`)
**Purpose**: Verify the public Cloudflare Tunnel endpoint can route traffic from the internet to the local vLLM server.  
**Expected Result**: HTTP 200 containing `qwen3-coder-30b`.

In [ ]:
import urllib.request
import json

public_models_url = f"{tunnel_url}/v1/models"
print(f"Testing public endpoint: {public_models_url}")

req = urllib.request.Request(public_models_url)
with urllib.request.urlopen(req, timeout=10) as resp:
    pub_data = json.loads(resp.read().decode())

print(f"Status Code: HTTP {resp.status}")
print("Available Models:", [m["id"] for m in pub_data.get("data", [])])
print("Public Ingress Verification: PASS")

## 15. Public Inference Test (`POST /v1/chat/completions`)
**Purpose**: Send a full chat completion request through the public HTTPS tunnel.  
**Expected Result**: Successful response with prompt echo from Qwen.

In [ ]:
import urllib.request
import json

payload = {
    "model": "qwen3-coder-30b",
    "messages": [
        {"role": "user", "content": "Reply with only: REMOTE_VERIFIED_SUCCESS"}
    ],
    "temperature": 0.0,
    "max_tokens": 16
}

req = urllib.request.Request(
    f"{tunnel_url}/v1/chat/completions",
    data=json.dumps(payload).encode("utf-8"),
    headers={"Content-Type": "application/json"}
)

with urllib.request.urlopen(req, timeout=45) as resp:
    pub_res = json.loads(resp.read().decode())

pub_content = pub_res["choices"][0]["message"]["content"].strip()
print("Response via Cloudflare Tunnel:")
print(f"  Content: '{pub_content}'")
assert "REMOTE_VERIFIED_SUCCESS" in pub_content, f"Unexpected response: {pub_content}"
print("\nPublic End-to-End Chat Completion: PASS")

## 16. Relay Gateway Configuration Snippet
**Purpose**: Generate the exact environment configuration to copy into your local Relay `.env` file on your development machine.

In [ ]:
relay_env_snippet = f'''
# ==============================================================================
# Add or update these lines in your local Relay .env file:
# ==============================================================================
QWEN_BASE_URL={tunnel_url}/v1
QWEN_MODEL=qwen3-coder-30b
# QWEN_API_KEY= (leave unset or blank for this unauthenticated backend)
# ==============================================================================
'''

print(relay_env_snippet)
print("After updating .env, start Relay on your local machine with:")
print("  pnpm dev")

## 17. Runtime Status & Log Monitoring
**Purpose**: Display process status, PIDs, active endpoints, and convenient commands for tailing logs.

In [ ]:
import os

def read_pid(path):
    if os.path.exists(path):
        with open(path) as f:
            return f.read().strip()
    return "Not Found"

print("========================================================")
print(" RELAY QWEN/vLLM RUNTIME STATUS")
print("========================================================")
print(f"vLLM Process PID:       {read_pid('/kaggle/working/vllm.pid')}")
print(f"cloudflared PID:        {read_pid('/kaggle/working/cloudflared.pid')}")
print(f"Local Endpoint:         http://127.0.0.1:8000/v1")
print(f"Public Tunnel Endpoint: {tunnel_url}/v1")
print(f"vLLM Log File:          /kaggle/working/vllm_server.log")
print(f"cloudflared Log File:   /kaggle/working/cloudflared.log")
print("========================================================")
print("\nUseful Commands:")
print("  !tail -n 50 /kaggle/working/vllm_server.log")
print("  !tail -n 30 /kaggle/working/cloudflared.log")
print("  !nvidia-smi")

## 18. Safe Shutdown Procedure
**Purpose**: Stop both the Cloudflare Tunnel and the vLLM server processes, verify they terminate, and confirm that all GPU memory is freed back to Kaggle.  
**When to Run**: Run this cell when you are finished testing to release compute resources.

In [ ]:
import os
import signal
import time
import subprocess

print("Initiating safe shutdown of backend processes...")

# 1. Stop Cloudflare Tunnel
if os.path.exists("/kaggle/working/cloudflared.pid"):
    try:
        with open("/kaggle/working/cloudflared.pid") as f:
            cf_pid = int(f.read().strip())
        print(f"Stopping cloudflared (PID {cf_pid})...")
        os.kill(cf_pid, signal.SIGTERM)
    except Exception as e:
        print(f"Notice: {e}")
    os.remove("/kaggle/working/cloudflared.pid")

# 2. Stop vLLM server
if os.path.exists("/kaggle/working/vllm.pid"):
    try:
        with open("/kaggle/working/vllm.pid") as f:
            vllm_pid = int(f.read().strip())
        print(f"Stopping vLLM (PID {vllm_pid})...")
        os.kill(vllm_pid, signal.SIGTERM)
        time.sleep(3)
        try:
            os.kill(vllm_pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
    except Exception as e:
        print(f"Notice: {e}")
    os.remove("/kaggle/working/vllm.pid")

# Kill any remaining worker sub-processes
subprocess.run(["pkill", "-f", "vllm"], capture_output=True)
time.sleep(3)

print("Shutdown completed.")
print("\nFinal GPU Memory Status:")
subprocess.run(["nvidia-smi", "--query-gpu=index,memory.used,memory.free", "--format=csv,noheader"])